In [146]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [147]:
import numpy as np
from sklearn.linear_model import Lasso
import matplotlib.pyplot as plt

import utils
import parametric
import CoRT_builder
import json
import datetime
import os

# Uniform

In [148]:
def parametric_uniform_test(n_target, n_source, p, K, Ka, h, lamda, alpha, T, cnt):
    s_vector = [0] * cnt
    s = len(s_vector)
    CoRT_model = CoRT_builder.CoRT(alpha=lamda)

    target_data, source_data = CoRT_model.gen_data(n_target, n_source, p, K, Ka, h, s_vector, s, "AR")
    similar_source_index = CoRT_model.find_similar_source(n_target, K, target_data, source_data, T=T, verbose=False)
    X_combined, y_combined = CoRT_model.prepare_CoRT_data(similar_source_index, source_data, target_data)

    model = Lasso(alpha=lamda, fit_intercept=False, tol=1e-10, max_iter=10000000)
    model.fit(X_combined, y_combined.ravel())
    beta_hat_target = model.coef_[-p:]

    active_indices = np.array([i for i, b in enumerate(beta_hat_target) if b != 0])

    if len(active_indices) == 0:
        print("Lasso selected no features. Skipping.")
        return

    j = np.random.choice(len(active_indices))

    X_target = target_data["X"]
    y_target = target_data["y"]
    X_active, X_inactive = utils.get_active_X(beta_hat_target, X_target)

    etaj, etajTy = utils.construct_test_statistic(y_target, j, X_active)

    Sigma = np.eye(n_target)
    b_global = Sigma @ etaj @ np.linalg.pinv(etaj.T @ Sigma @ etaj)
    a_global = (Sigma - b_global @ etaj.T) @ y_target

    folds = utils.split_target(T, X_target, y_target, n_target)

    tn_sigma = (np.sqrt(etaj.T @ Sigma @ etaj)).item()

    z_min = -20  * tn_sigma
    z_max = 20 * tn_sigma

    interval = {}
    print(f"Starting with = {z_min}, z_max = {z_max}")
    for k in range(K):
        for t in range(T):
            interval[(k, t)] = parametric.find_interval(t, k, z_min, z_max, folds, source_data, a_global, b_global, lamda, K, T)

    z_k = z_min
    z_list = [z_k]
    Az_list = []

    while z_k < z_max:
        mn, similar_source_index = parametric.find_similar_source(z_k, z_max, interval, K, T)
        target_data_current = {"X": X_target, "y": a_global + z_k * b_global}
        X_combined_new, y_combined_new = CoRT_model.prepare_CoRT_data(similar_source_index, source_data, target_data_current)
        L_CoRT, R_CoRT, Az = parametric.get_Z_CoRT(X_combined_new, similar_source_index, lamda, a_global, b_global, source_data, z_k)


        current_num_sources = len(similar_source_index)
        offset = p * current_num_sources
        Az_target_current = np.array([idx - offset for idx in Az if idx >= offset])
        Az_list.append(Az_target_current)
        mn = min(mn, R_CoRT)

        z_k = max(z_k, mn) + 1e-5

        if (z_k >= z_max):
            z_list.append(z_max)
            break
        else:
            z_list.append(z_k)

    optim_p_value = utils.pivot(active_indices, Az_list, z_list, etaj, etajTy, 0, Sigma)

    print(f"optim_p_value = {optim_p_value}")
    return optim_p_value 

In [149]:
n_target = 50
n_source = 20
p = 100
K = 5 
Ka = 3
h = 30 
lamda = 0.05
alpha = 0.05
iteration = 1
T = 5
cnt = 25
s_vector = [0] * cnt
s = len(s_vector)
CoRT_model = CoRT_builder.CoRT(alpha=lamda)
owner = "Nhodeptrai"
p_values = []
iteration = 100

for i in range(iteration):
    p_value = parametric_uniform_test(n_target, n_source, p, K, Ka, h, lamda, alpha, T, cnt)
    p_values.append(p_value)

results = {
    "timestamp": datetime.datetime.now().isoformat(),
    "owner": owner,
    "params": {
        "n_target": n_target,
        "n_source": n_source,
        "p": p, 
        "K": K,
        "Ka": Ka,
        "h": h, 
        "lamda": lamda,
        "alpha": alpha,
        "iteration": iteration,
        "T": T,
        "cnt": cnt
    },
    "p_values": p_values,
    "len": len(p_values)
}

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

output_filename = "p-100_uniform_test.json"

if not os.path.exists(output_filename):
    all_results = [results]
else: 
    with open(output_filename, "r", encoding="utf-8") as f:
        all_results = json.load(f)
        all_results.append(results)

total_p_values = 0
for r in all_results:
    total_p_values += r["len"]
print(f"Total p-value: {total_p_values}")

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(all_results, f, cls=NumpyEncoder, indent=4)

print(f"Đã lưu kết quả thành công vào file: {output_filename}")

Starting with = -5.323028193211357, z_max = 5.323028193211357
optim_p_value = 0.5360203065469054
Starting with = -3.4598962087648926, z_max = 3.4598962087648926


KeyboardInterrupt: 